In [ ]:
/**
 * DATABASE EVOLUTION — Zero-Downtime Schema Changes
 * 1.  Why Schema Changes Are Dangerous ... Tables are large, traffic is live
 * 2.  What Does Downtime Mean? ........... Full vs partial availability
 * 3.  The Safe Pattern ................... Expand → Backfill → Switch → Contract
 * 4.  A Simple Example ................... Adding a payment_mode column
 * 5.  Backfills Are the Real Danger ...... Why migration takes days, not minutes
 * 6.  Old and New Versions Must Coexist .. Backward compatibility is everything
 * 7.  Safe vs Risky Changes .............. What you can do, what you should avoid
 * 8.  When Downtime Is Still Acceptable .. Honest tradeoffs for smaller systems
 * 9.  Golden Rules ....................... Key principles to remember
 */

In [ ]:
/**
 * Part-1: WHY SCHEMA CHANGES ARE DANGEROUS
 */

In [ ]:
/**
 * Tables Are Large, Traffic Is Live
 * ────────────────────────────────────
 *
 *   Chhoti scale pe schema changes easy lagte hain.
 *   Badi scale pe DANGEROUS ho jaate hain — billions of rows + 24/7 live traffic.
 *   Schema change karna = DB + backend + old app + new app + existing data sab ke
 *   beech ka CONTRACT change karna.
 *
 * AS PRODUCTS EVOLVE, SCHEMAS MUST EVOLVE TOO:
 *   → Add a new column
 *   → Add a new table
 *   → Relax a constraint
 *   → Introduce new read/write flow
 *   → Migrate data format
 *   → Split data differently
 *
 * AT SMALL SCALE: Schema changes feel easy.
 * AT SCALE: Schema changes become DANGEROUS because:
 *   → Tables are LARGE (billions of rows)
 *   → Traffic is LIVE (24/7 reads and writes)
 *   → Old app versions may still be running
 *   → Cached clients may still call old APIs
 *   → Background jobs depend on the current schema
 *
 * THE REAL GOAL:
 *   Not just "change the schema."
 *   "Change the schema WITHOUT BREAKING PRODUCTION TRAFFIC."
 *
 * RISKY VS SAFE SCHEMA CHANGES:
 *   → Add a column          (safe)
 *   → Rename a column       (dangerous)
 *   → Change a data type    (dangerous)
 *   → Make nullable NOT NULL (dangerous)
 *   → Split one field into multiple (dangerous)
 *   → Change primary key    (very dangerous)
 *
 *   These can break: live writes, live reads, old code paths,
 *   cached clients, and background jobs.
 */

In [ ]:
/**
 * Part-2: WHAT DOES "DOWNTIME" ACTUALLY MEAN?
 */

In [ ]:
/**
 * Full vs Partial vs Zero Downtime
 * ──────────────────────────────────
 *
 *   "Downtime" ka matlab poora app band hona zaroori nahi.
 *   Partial downtime bhi hoti hai — jaise bank maintenance mein
 *   login karo par transfer nahi kar sakte.
 *
 * TYPES OF DOWNTIME:
 *
 *   FULL DOWNTIME:
 *   → Entire service becomes unavailable
 *   → No requests can be processed
 *   → Users see errors or maintenance pages
 *
 *   PARTIAL DOWNTIME:
 *   → Only specific capabilities are unavailable
 *   → Example during banking maintenance:
 *       You can still log in             ✓
 *       You can still view balances      ✓
 *       Money transfers are disabled     ✗
 *
 *   ZERO DOWNTIME:
 *   → No noticeable interruption to users
 *   → All critical paths continue working
 *   → Old and new versions COEXIST during transition
 *
 * THE GOAL OF ZERO-DOWNTIME MIGRATION:
 *   → Avoid FULL downtime
 *   → Minimize PARTIAL downtime as much as possible
 */

In [ ]:
/**
 * Part-3: THE SAFE PATTERN
 */

In [ ]:
/**
 * Expand → Backfill → Switch → Contract
 * ────────────────────────────────────────
 *
 *   Ye proven 4-phase pattern hai zero-downtime schema changes ke liye.
 *   Ek hi baar sab mat karo — gradually phases mein karo.
 *
 * PHASE 1: EXPAND — Add new schema in a compatible way
 *   → Add a new NULLABLE column (NOT NOT NULL!)
 *   → Add a new table
 *   → Add a new index concurrently
 *   → Introduce new code paths WITHOUT removing old ones
 *
 *   KEY PRINCIPLE: Keep the old path working.
 *   Do not delete old behavior immediately.
 *
 * PHASE 2: BACKFILL — Populate old data into new structure
 *   → Read old rows
 *   → Compute the new representation
 *   → Write it back GRADUALLY
 *
 *   This is the MOST DANGEROUS phase.
 *   Must be rate-limited, batched, and monitored.
 *
 * PHASE 3: SWITCH — Move reads/writes to new path
 *   → New writes start populating the new field
 *   → Reads begin using the new field
 *   → New app logic depends on the new structure
 *
 *   This switch must be GRADUAL, not flipped in one second.
 *
 * PHASE 4: CONTRACT — Remove the old path
 *   → Stop writing to old column
 *   → Stop reading from old format
 *   → Remove old code
 *   → Finally DROP old schema pieces
 *
 *   This is the LAST step, not the first!
 *
 * VISUAL TIMELINE:
 *
 *   Expand ──────────────────────────────────────────────────────────►
 *        │
 *        └──► Backfill (gradual, rate-limited)
 *                    │
 *                    └──► Switch (gradual, monitored)
 *                              │
 *                              └──► Contract (cleanup)
 */

In [ ]:
/**
 * Part-4: A SIMPLE EXAMPLE — Adding a payment_mode column
 */

In [ ]:

/**
 * Payment Mode Column Add Karna — Step by Step
 * ──────────────────────────────────────────────
 *
 *   payments table mein pehle payment_mode column nahi tha.
 *   Ab chahiye ki user apna payment history mein mode dekh sake.
 *   Ye 4-phase process follow karo bina downtime ke.
 *
 * BEFORE MIGRATION:
 *   payments table: id, amount, user_id, status, created_at
*/

/**
 * STEP 1: EXPAND — Column safely add karo
 */


ALTER TABLE payments ADD COLUMN payment_mode VARCHAR(20) NULL;
-- WHY NULLABLE?
-- Old code paths still insert rows without this field.
-- Making it NOT NULL immediately → breaks old code!
-- NULLABLE = backward compatible ✅

-- Current state after EXPAND:
-- New column exists
-- All existing rows have payment_mode = NULL
-- Old code still works (doesn't know about the column)
-- New code can start writing to it


/**
 * STEP 2: BACKFILL — Old rows mein correct values daalo
 *
 *   Old rows need payment_mode populated.
 *   Maano sab old rows 'cash_on_delivery' type ki thi.
 *   IMPORTANT: Batches mein karo — ek saath nahi!
 */


-- ❌ BAD: Ek saath sab rows update (table lock!)
UPDATE payments
SET payment_mode = 'cash_on_delivery'
WHERE payment_mode IS NULL;

-- ✅ GOOD: Batches mein karo (1000 rows at a time)
-- Process 1000 rows → wait → repeat
UPDATE payments
SET payment_mode = 'cash_on_delivery'
WHERE payment_mode IS NULL
LIMIT 1000;
-- Wait 1 second → Repeat until done


/**
 * STEP 3: SWITCH — New column use karna shuru karo
 *
 *   → Write path update: sab new payments write payment_mode
 *   → Read path update: payment history reads payment_mode
 *   → New code deploy karo jo column use kare
 *
 *   During this phase:
 *     New code writes payment_mode ✓
 *     Backfill still catching up old rows
 *     Old code paths still work if any remain
*/

/**
 * STEP 4: CONTRACT — Old path clean up karo
 */


-- Once everything is stable and all rows have values:
-- Make payment_mode NOT NULL agar zaroorat ho
ALTER TABLE payments ALTER COLUMN payment_mode SET NOT NULL;
-- (Only after ALL rows have values — warna ERROR!)

-- Remove old code that doesn't use payment_mode

In [ ]:
/**
 * Part-5: BACKFILLS ARE THE REAL DANGER
 */

In [ ]:
/**
 * Migration Days Kyun Lagti Hai — Minutes Nahi
 * ──────────────────────────────────────────────
 *
 *   Log sochte hain schema migration ka risk column add karna ya code change hai.
 *   ASLI DANGER hai BACKFILL — billions of rows touch karna.
 *   Ye production pe spike la sakta hai agar sahi se na karo.
 *
 * MANY PEOPLE THINK SCHEMA MIGRATION DANGER IS:
 *   → Adding the column
 *   → Writing the migration script
 *   → Changing the code
 *
 * THE REAL DANGER IS OFTEN THE BACKFILL.
 *
 * WHY BACKFILLS ARE DANGEROUS:
 *   When you touch MANY rows:
 *   → Lots of writes
 *   → More I/O
 *   → More locking pressure
 *   → More replication lag
 *   → More disk activity
 *   → More cache churn
 *   → Possible latency SPIKES for production traffic
 *
 * REAL-WORLD EXAMPLE:
 *   Adding a column?     → Easy — milliseconds.
 *   Backfilling 1 billion rows? → That's DAYS or WEEKS.
 *
 * HOW BACKFILLS SHOULD BE DONE:
 *   ✅ Rate-limited
 *   ✅ Done in BATCHES (100-1000 rows at a time)
 *   ✅ Done in the BACKGROUND
 *   ✅ MONITORED carefully
 *   ✅ PAUSABLE if production health degrades
 *   ✅ RESUMABLE when safe
 *
 * DO NOT:
 *   ❌ UPDATE table SET column = value;  (all at once!)
 *
 *   This can:
 *   → Lock the table for a long time
 *   → Spike replication lag
 *   → Cause timeouts
 *   → CRASH production
 *
 * AT VERY LARGE SCALE:
 *   Backfills can take DAYS or WEEKS.
 *   And that's OKAY.
 *   A slow safe rollout is BETTER than a fast outage.
 */

In [ ]:
/**
 * Part-6: OLD AND NEW VERSIONS MUST COEXIST
 */

In [ ]:
/**
 * Backward Compatibility Is Everything
 * ──────────────────────────────────────
 *
 *   Production mein ek saath old aur new versions chal rahi hoti hain.
 *   Agar tum achanak koi field ya API hatao — old clients IMMEDIATELY BREAK.
 *   Isliye multiple deployment phases chahiye.
 *
 * REALITY IN PRODUCTION:
 *   → Backend v1 exists
 *   → Frontend/mobile apps still use v1
 *   → Backend v2 is being rolled out
 *   → Schema is changing underneath
 *
 *   If you suddenly remove an old field or old API path,
 *   OLD CLIENTS MAY BREAK IMMEDIATELY.
 *
 * KEY PRINCIPLE:
 *   Old clients should STILL WORK.
 *   New clients should ALSO WORK.
 *   Old schema expectations should NOT instantly disappear.
*/

/**
 * HOW TO MAINTAIN BACKWARD COMPATIBILITY:
 *
 *   1. ADDITIVE changes are safe:
 *      → Adding a new column (NULLABLE)
 *      → Adding a new table
 *      → Adding a new index
 *
 *   2. DESTRUCTIVE changes need careful choreography:
 *      → Renaming a column
 *      → Changing data type
 *      → Dropping a column
 *
 *   3. The DUAL-WRITE PATTERN:
 *      → Write to BOTH old and new structures temporarily
 *      → Old clients read from old structure
 *      → New clients read from new structure
 *      → Eventually switch all reads, then stop writes to old
*/

/**
 * EXAMPLE: Renaming a column with zero downtime
 */


-- Phase 1: Add new column, start dual-writing
ALTER TABLE users ADD COLUMN new_name VARCHAR(100);
-- App writes to BOTH old_name AND new_name

-- Phase 2: Backfill old rows
UPDATE users SET new_name = old_name WHERE new_name IS NULL;
-- (in batches, of course)

-- Phase 3: Switch reads
-- Deploy app that reads from new_name instead of old_name

-- Phase 4: Stop dual-write
-- App stops writing to old_name

-- Phase 5: Clean up
ALTER TABLE users DROP COLUMN old_name;


/**
 * This is ZERO-DOWNTIME but takes MULTIPLE DEPLOYMENT STEPS.
 * Ye time lagta hai — par production safe rehta hai.
 */

In [ ]:
/**
 * Part-7: SAFE VS RISKY CHANGES
 */

In [ ]:
/**
 * Kya Safe Hai, Kya Risky Hai
 * ─────────────────────────────
 *
 *   Additive changes usually safe hote hain — backward compatible.
 *   Destructive changes risky hote hain — existing code break kar sakte hain.
 *
 * SAFER CHANGES (near-zero downtime):
 *
 *   ✅ Adding a NULLABLE column
 *   ✅ Adding a new table
 *   ✅ Adding an index CONCURRENTLY
 *   ✅ Relaxing constraints (making NOT NULL nullable)
 *   ✅ Increasing VARCHAR length
 *   ✅ Adding a new foreign key (without validation initially)
 *
 *   WHY SAFER? They are ADDITIVE and BACKWARD-COMPATIBLE.
 *   They don't invalidate existing code or data.
 *
 * RISKIER CHANGES (need careful choreography):
 *
 *   ❌ Dropping a column
 *   ❌ Renaming a column
 *   ❌ Changing a column's data type
 *   ❌ Making a nullable column NOT NULL
 *   ❌ Splitting one column into multiple columns
 *   ❌ Changing primary key design
 *   ❌ Changing a column's default value to something incompatible
 *
 *   WHY RISKIER? They are DESTRUCTIVE or BREAK existing assumptions.
 *   They can break:
 *   → App code that expects old name/type
 *   → Old queries that worked before
 *   → Background jobs that rely on old schema
 *   → ORM mappings that expect certain columns
 *
 * FOR RISKY CHANGES:
 *   → Use Expand → Backfill → Switch → Contract pattern
 *   → Maintain backward compatibility through multiple releases
 *   → Coordinate with all teams that depend on the schema
 *   → Have rollback plans
 */

In [ ]:
/**
 * Part-8: WHEN DOWNTIME IS STILL ACCEPTABLE
 */

In [ ]:
/**
 * Honest Tradeoffs for Smaller Systems
 * ──────────────────────────────────────
 *
 *   Zero-downtime migration complex aur expensive hai.
 *   Chhote systems ke liye scheduled maintenance window perfectly valid hai.
 *   Apni requirements ke hisaab se honest tradeoff karo.
 *
 * WHEN DOWNTIME IS ACCEPTABLE:
 *   → Low-traffic windows (e.g., 3 AM Sunday)
 *   → Early-stage products with few users
 *   → Internal tools where downtime is communicated
 *   → Small systems where user impact is limited
 *   → One-time migrations where zero-downtime complexity > benefit
 *
 * REAL-WORLD EXAMPLES:
 *
 *   Young product with 100 daily users:
 *   → "We'll take 5 minutes of downtime at 2 AM"
 *   → Perfectly reasonable!
 *
 *   Cloud provider with 99.99% SLA:
 *   → Must invest heavily in zero-downtime migrations
 *   → Downtime is extremely expensive
 *
 *   Banking system:
 *   → Some downtime for specific flows may be acceptable
 *   → But critical money movement must stay up
 *
 * THE HONEST TRUTH:
 *   Zero-downtime migrations are COMPLEX AND EXPENSIVE.
 *   For many systems, a scheduled maintenance window is
 *   a PERFECTLY VALID choice.
 *
 *   The key is:
 *   → Know your uptime requirements
 *   → Know your user expectations
 *   → Be honest about tradeoffs
 *   → Plan for the complexity that zero-downtime requires
 *
 *   Large platforms invest heavily in zero-downtime strategies
 *   because downtime is UNACCEPTABLE for them.
 *   Smaller systems may NOT need that level of investment.
 */
